# Query Explorer
Select a category and query ID to inspect the source recipe, ground-truth strong list, and hybrid search results.

In [13]:
import pandas as pd
from pathlib import Path

BASE      = Path("/Users/alice/project/semantic_search/search_engine/pipeline")
EVAL_DIR  = BASE / "03_evaluate_embeddings"
DATA_DIR  = BASE / "02_synthesize_data"
PARQUET   = BASE / "01_process_data/cleaned/recipe_summaries.parquet"
OUT_DIR   = Path("/Users/alice/project/semantic_search/search_engine/pipeline/04_examples_generation")

recipes = pd.read_parquet(PARQUET)
recipes["flow_id"] = recipes["flow_id"].astype(int)

cat_datasets = {
    cat: pd.read_csv(DATA_DIR / f"category{cat}_dataset.csv")
    for cat in [1, 2, 3]
}

# Hybrid (FTS/english + Qwen3-Embedding-8B+instruct)
hybrid_results = {
    cat: pd.read_csv(EVAL_DIR / f"hybrid_english+Qwen_Qwen3_Embedding_8B+instruct_category{cat}_k5.csv")
    for cat in [1, 2, 3]
}

# Dense: Qwen3-Embedding-8B+instruct (one file, all categories)
_dense_all = pd.read_csv(EVAL_DIR / "eval_results_Qwen_Qwen3_Embedding_8B+instruct_k5.csv")
dense_results = {
    cat: _dense_all[_dense_all["category"] == f"Category {cat}"].reset_index(drop=True)
    for cat in [1, 2, 3]
}

# Keyword ILIKE (per-category files)
ilike_results = {
    cat: pd.read_csv(EVAL_DIR / f"fts_category{cat}_k5.csv")
    for cat in [1, 2, 3]
}

# FTS English / pgfts (per-category files)
pgfts_results = {
    cat: pd.read_csv(EVAL_DIR / f"pgfts_english_category{cat}_k5.csv")
    for cat in [1, 2, 3]
}

def get_summary(uid: str) -> str:
    parts = uid.split("_")
    try:
        flow_id = int(parts[1])
        row = recipes[recipes["flow_id"] == flow_id]
        if not row.empty:
            return row["recipe_summary_without_comment"].values[0]
    except (IndexError, ValueError):
        pass
    return "(not in local parquet)"

def parse_uid_list(val) -> list:
    if pd.isna(val) or not str(val).strip():
        return []
    return [u.strip() for u in str(val).split(",") if u.strip()]

print("Loaded.")
for cat, df in cat_datasets.items():
    print(f"  Category {cat}: {len(df)} queries")


Loaded.
  Category 1: 50 queries
  Category 2: 50 queries
  Category 3: 49 queries


In [20]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CATEGORY = 3          # 1, 2, or 3
QUERY_ID = "8621_c3q20"
# ─────────────────────────
# ────────────────────────────────────────────────────

SEP = "=" * 70

dataset = cat_datasets[CATEGORY]
q_row   = dataset[dataset["query_id"] == QUERY_ID]

if q_row.empty:
    print(f"Query ID '{QUERY_ID}' not found in Category {CATEGORY}.")
    print("Available IDs:", list(dataset["query_id"]))
else:
    q           = q_row.iloc[0]
    strong_uids = parse_uid_list(q["strong_list"])
    source_fid  = int(q["source_flow_id"])

    print(SEP)
    print(f"Category {CATEGORY}  |  Query ID: {QUERY_ID}")
    print(SEP)
    print(f"\nQUERY:\n  {q['query']}")
    print(f"\nSTRONG LIST ({len(strong_uids)} recipes):")
    for uid in strong_uids:
        print(f"  {uid}")

    print(f"\n{SEP}")
    print(f"SOURCE RECIPE  (flow_id={source_fid})")
    print(SEP)
    src_row = recipes[recipes["flow_id"] == source_fid]
    print(src_row["recipe_summary_without_comment"].values[0] if not src_row.empty else "(not in local parquet)")


Category 3  |  Query ID: 8621_c3q20

QUERY:
  If the workato_db_table get_records action changes (table_id/order_by_field_id/filters/continuation_token), which recipes will be affected?

STRONG LIST (6 recipes):
  136763_23098236_v2
  136763_61448486_v1
  136763_61818873_v14
  1672048_15022103_v41
  50737_56093655_v83
  8621_63733901_v36

SOURCE RECIPE  (flow_id=63733901)
Connectors: csv_parser, workato_db_table, workato_genie, workato_transformations, workato_variable
Steps:
- trigger: workato_genie / start_workflow
  - action: workato_variable / declare_list
    fields: list_item_schema_json, name
  - action: workato_variable / declare_variable
    fields: variables
  - repeat [loop]
    - if [and: blank, blank]
      - action: workato_db_table / get_records
        fields: order_direction, limit, table_id, order_by_field_id, continuation_token
      - action: workato_variable / insert_to_list_batch
        fields: location, name, list_items
      - elsif [and: present, blank]
      

In [21]:
import re as _re

def extract_keywords(text: str) -> list[str]:
    """Extract underscore tokens (technical keywords) from text."""
    tokens = _re.findall(r'[A-Za-z0-9][A-Za-z0-9_]*', text.lower())
    return list(dict.fromkeys(t for t in tokens if '_' in t))  # deduplicated, ordered

def highlight(text: str, keywords: list[str]) -> str:
    """Wrap each keyword occurrence in **bold** (case-insensitive)."""
    for kw in sorted(keywords, key=len, reverse=True):  # longest first to avoid partial matches
        text = _re.sub(
            _re.escape(kw),
            lambda m: f"**{m.group(0)}**",
            text,
            flags=_re.IGNORECASE,
        )
    return text

def show_method(label, result_df, query_id, strong_set):
    SEP = "=" * 70
    row = result_df[result_df["query_id"] == query_id]
    if row.empty:
        print(f"{SEP}\n{label}: no result found\n")
        return []
    h = row.iloc[0]
    retrieved = parse_uid_list(h["retrieved"])
    print(f"\n{SEP}")
    print(label)
    print(SEP)
    for uid in retrieved:
        print(f"  {uid}")
    return retrieved

q_row = cat_datasets[CATEGORY][cat_datasets[CATEGORY]["query_id"] == QUERY_ID]
if q_row.empty:
    print(f"Run cell 2 first — QUERY_ID '{QUERY_ID}' not found.")
else:
    q           = q_row.iloc[0]
    strong_uids = parse_uid_list(q["strong_list"])
    strong_set  = set(strong_uids)
    source_fid  = int(q["source_flow_id"])

    query_text = q['query']
    keywords   = extract_keywords(query_text)

    methods = [
        ("hybrid retrieval: fts-english+Qwen3-Embedding-8B+instruct", hybrid_results[CATEGORY]),
        ("dense/Qwen3-Embedding-8B+instruct",               dense_results[CATEGORY]),
        ("keyword/ilike",                                    ilike_results[CATEGORY]),
        ("pgfts/english",                                    pgfts_results[CATEGORY]),
    ]

    for label, df in methods:
        show_method(label, df, QUERY_ID, strong_set)

    # ── Save markdown ─────────────────────────────────────────────────────────
    src_row     = recipes[recipes["flow_id"] == source_fid]
    src_summary = src_row["recipe_summary_without_comment"].values[0] if not src_row.empty else "(not in local parquet)"

    def method_section(label, result_df, query_id, keywords):
        row = result_df[result_df["query_id"] == query_id]
        lines = [f"## {label}", ""]
        if row.empty:
            return lines + ["No result found.", ""]
        retrieved = parse_uid_list(row.iloc[0]["retrieved"])
        for uid in retrieved:
            summary = highlight(get_summary(uid), keywords)
            lines += [f"### `{uid}`", "", summary, ""]
        return lines

    md_lines = [
        f"# Example - {QUERY_ID}", "",
        "---", "",
        "## Query", "",
        f"> {highlight(query_text, keywords)}", "",
        "---", "",
        f"## Ground-Truth Strong List ({len(strong_uids)} recipes)", "",
    ]
    for uid in strong_uids:
        md_lines.append(f"- `{uid}`")
    md_lines += ["", "---", ""]

    for label, df in methods:
        md_lines += method_section(label, df, QUERY_ID, keywords)
        md_lines += ["---", ""]

    md_lines += [
        f"## Source Recipe (flow\_id={source_fid})", "",
        highlight(src_summary, keywords), "",
    ]

    out_path = OUT_DIR / f"query_explorer_cat{CATEGORY}_{QUERY_ID}.md"
    out_path.write_text("\n".join(md_lines))
    print(f"\nSaved -> {out_path}")
    print(f"Keywords detected ({len(keywords)}): {keywords}")



hybrid retrieval: fts-english+Qwen3-Embedding-8B+instruct
  384059_52352657_v71
  804586_47150941_v29
  602425_66687816_v1
  136763_64449339_v4
  973497_53963798_v104

dense/Qwen3-Embedding-8B+instruct
  804586_47150941_v29
  384059_52352657_v71
  69943_63763557_v15
  136763_52738101_v5
  804586_49506728_v10

keyword/ilike
  136763_50417983_v1
  1873346_47040880_v7
  804586_51482000_v7
  8621_63733901_v36

pgfts/english
  50737_56890803_v18
  69943_59239995_v61
  804586_47168222_v59
  602425_66687816_v1
  69943_60939422_v2

Saved -> /Users/alice/project/semantic_search/search_engine/pipeline/04_examples_generation/query_explorer_cat3_8621_c3q20.md
Keywords detected (5): ['workato_db_table', 'get_records', 'table_id', 'order_by_field_id', 'continuation_token']


<>:88: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
<>:88: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
/var/folders/f7/z9rtwn457xx81k6mg0pbwht00000gn/T/ipykernel_20522/1451618794.py:88: SyntaxWarning: "\_" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\_"? A raw string is also an option.
  f"## Source Recipe (flow\_id={source_fid})", "",
